# Using GPErks training

In [ ]:
import numpy as np
import torch
import sys
sys.path.append('../simulation_toolbox/')
from GPErks_modified.log.logger import get_logger
from GPErks_modified.utils.random import set_seed
from sklearn.model_selection import train_test_split
from GPErks_modified.gp.data.dataset import Dataset
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.means import LinearMean
from gpytorch.kernels import RBFKernel, ScaleKernel
from torchmetrics import MeanSquaredError, R2Score
from GPErks_modified.gp.experiment import GPExperiment
from GPErks_modified.perks.cross_validation import KFoldCrossValidation
from GPErks_modified.train.early_stop import GLEarlyStoppingCriterion
from GPErks_modified.train.emulator import GPEmulator
from GPErks_modified.train.early_stop import NoEarlyStoppingCriterion

log = get_logger()
seed = 8
set_seed(seed)

In [ ]:
import os

# load dataset
mesh = 3
scenario = 42
basefolder=f"/path/to/data/Elements/HCM/{mesh}/scenarios/{scenario}"
emulators_folder_base = F"{basefolder}/output/emulators/"
feature_idx = 0


X_all =  np.loadtxt(f"{basefolder}/data/X.txt", dtype=float)
mask =  np.loadtxt(f"{basefolder}/output/output_mask.txt", dtype=float)
mask = mask.astype(bool)


input_masked = X_all[:mask.shape[0]]  # Trim X_all to match the size of the mask
X_ = input_masked[mask]
y_all = np.loadtxt(f"{basefolder}/data/Y.txt", dtype=float)

y_feature = y_all[:,feature_idx]
y_ = y_feature[mask]

with open(f"{basefolder}/data/xlabels.txt", "r") as f:
        x_labels = f.read().splitlines()

with open(f"{basefolder}/data/ylabels.txt", "r") as f:
        y_labels = f.read().splitlines()

emulators_folder = f"{emulators_folder_base}/{y_labels[feature_idx]}"

# split dataset in training and validation sets
X, X_test, y, y_test = train_test_split(
    X_,
    y_,
    test_size=0.2,
    random_state=seed
)


In [ ]:
from GPErks_modified.utils.metrics import IndependentStandardErrorMetric


dataset = Dataset(
    X,
    y,
    x_labels=x_labels,
    y_label=y_labels[feature_idx]
)

likelihood = GaussianLikelihood()
mean_function = LinearMean(input_size=dataset.input_size)
kernel = ScaleKernel(RBFKernel(ard_num_dims=dataset.input_size))
metrics = [IndependentStandardErrorMetric(), MeanSquaredError(), R2Score()] # Current 3 metrics. The IndependentStandardErrorMetric() is a custom one that required GPErks modification

experiment = GPExperiment(
    dataset,
    likelihood,
    mean_function,
    kernel,
    n_restarts=3,
    metrics=metrics,
    seed=seed,
    learn_noise=True
)

In [ ]:
device = "cpu"
devices = [device]
kfcv = KFoldCrossValidation(experiment, devices, n_splits=5, max_workers=1)

optimizer = torch.optim.Adam(experiment.model.parameters(), lr=0.1)
esc = GLEarlyStoppingCriterion(
    max_epochs=1000, alpha=0.1, patience=8
)
best_model_dct, best_train_stats_dct, test_scores_dct = kfcv.train(
    optimizer,
    esc,
    leftout_is_val=True
)

In [ ]:
from GPErks_modified.perks.inference import Inference
from GPErks_modified.train.snapshot import NeverSaveSnapshottingCriterion
from GPErks_modified.serialization.path import posix_path
from GPErks_modified.constants import (
    DEFAULT_TRAIN_SNAPSHOT_RESTART_TEMPLATE,
    DEFAULT_TRAIN_SNAPSHOT_EPOCH_TEMPLATE,
)

best_epochs = []
for i, bts in best_train_stats_dct.items():
    best_epochs.append( bts.best_epoch )

import sys 
os.makedirs(emulators_folder, exist_ok=True)

with open(f"{emulators_folder}/training_summary.txt", "w") as f:
        print("Best epoch {}\n".format(best_epochs))
        print("Test Scores Dictionary:")
        print(test_scores_dct)

dataset = Dataset(
    X,
    y,
    X_test=X_test,
    y_test=y_test,
    x_labels=x_labels,
    y_label=y_labels[0]
)

# Saving the training and test sets for reproducibility
np.savetxt(f"{emulators_folder}/X_train.txt",X,fmt="%g")
np.savetxt(f"{emulators_folder}/X_test.txt",X_test,fmt="%g")
np.savetxt(f"{emulators_folder}/y_train.txt",y,fmt="%g")
np.savetxt(f"{emulators_folder}/y_test.txt",y_test,fmt="%g")

likelihood = GaussianLikelihood()
mean_function = LinearMean(input_size=dataset.input_size)
kernel = ScaleKernel(RBFKernel(ard_num_dims=dataset.input_size))

experiment = GPExperiment(
    dataset,
    likelihood,
    mean_function,
    kernel,
    n_restarts=3,
    metrics=metrics,
    seed=seed,  # reproducible training
    learn_noise=True
)
device = "cpu"

emulator = GPEmulator(experiment, device)

optimizer = torch.optim.Adam(experiment.model.parameters(), lr=0.1)
max_epochs = int( np.mean(best_epochs) )  # making use of cross-validation knowledge
esc = NoEarlyStoppingCriterion(max_epochs)

os.makedirs(emulators_folder, exist_ok=True)

snpc = NeverSaveSnapshottingCriterion(
        posix_path(
            f"{emulators_folder}/",
            DEFAULT_TRAIN_SNAPSHOT_RESTART_TEMPLATE,
        ),
        DEFAULT_TRAIN_SNAPSHOT_EPOCH_TEMPLATE,
    )

best_model, best_train_stats = emulator.train(
    optimizer,
    esc,
    snapshotting_criterion=snpc
)

In [ ]:
# Saving setup for reproducibility.

experiment.save_to_config_file(f"{emulators_folder}/emulator.ini")

In [ ]:
import matplotlib.pyplot as plt
from GPErks_modified.constants import HEIGHT, WIDTH

def plot_inference(inference, savepath, figname):
    fig, axis = plt.subplots(1, 1, figsize=(2 * WIDTH, 2 * HEIGHT / 3))

    idx_sort = np.argsort(
        inference.y_pred_mean
    )  # let's sort predicted values for a better visualisation
    x = np.arange(len(idx_sort))

    ci = 1.96  # 95% confidence interval

    axis.scatter(
        x,
        inference.y_test[idx_sort],
        facecolors="none",
        edgecolors="C0",
        label="observed",
    )
    axis.scatter(
        x,
        inference.y_pred_mean[idx_sort],
        facecolors="C0",
        s=16,
        label="predicted",
    )
    axis.errorbar(
        x,
        inference.y_pred_mean[idx_sort],
        yerr=ci * inference.y_pred_std[idx_sort],
        c="C0",
        ls="none",
        lw=0.5,
        label=f"uncertainty ({ci} STD)",
    )

    axis.set_xticks([])
    axis.set_xticklabels([])
    axis.legend(loc="upper left")

    fig.tight_layout()
    plt.savefig(f"{savepath}/{figname}.png")
    plt.close()

In [ ]:
inference = Inference(emulator)

with open(f"{emulators_folder}/training_summary.txt", "a") as f:
	sys_out = sys.stdout
	sys.stdout = f
	print("\n*** Final GPE ***")
	inference.summary()
	sys.stdout = sys_out

plot_inference(inference=inference, savepath=f"{basefolder}/figures", figname=f"gpe_inference_{y_labels[feature_idx]}")

# Summary table

In [ ]:
import os
from fpdf import FPDF

def create_pdf_with_table(ylabels_path, emulators_folder, output_pdf_path):
    # Initialize PDF
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Arial", size=10)

    # Read ylabels
    with open(ylabels_path, "r") as f:
        ylabels = [line.strip() for line in f.readlines()]

    # Create header row
    header_row = ["Output"]
    first_output_folder = os.path.join(emulators_folder, ylabels[0])
    first_summary_file = os.path.join(first_output_folder, "training_summary.txt")

    # Parse the first file to determine additional columns
    with open(first_summary_file, "r") as f:
        lines = f.readlines()
        start_index = lines.index("*** Final GPE ***\n") + 2
        columns = [line.split()[0] for line in lines[start_index:]]
    header_row.extend(columns)

    # Add header row to PDF
    cell_width = 40  # Adjust as needed
    pdf.set_font("Arial", style="B", size=10)
    for header in header_row:
        pdf.cell(cell_width, 10, header[:12] + "..." if len(header) > 12 else header, border=1, align='C')
    pdf.ln()

    # Parse each output
    pdf.set_font("Arial", size=10)
    for ylabel in ylabels:
        output_folder = os.path.join(emulators_folder, ylabel)
        summary_file = os.path.join(output_folder, "training_summary.txt")

        if not os.path.exists(summary_file):
            raise FileNotFoundError(f"Training summary file not found for {ylabel}.")

        with open(summary_file, "r") as f:
            lines = f.readlines()

            # Parse the final GPE section
            start_index = lines.index("*** Final GPE ***\n") + 2
            gpe_scores = {}
            for line in lines[start_index:]:
                parts = line.split()
                metric_name = parts[0]
                score = float(parts[-1])  # The last part is the score
                gpe_scores[metric_name] = score

            # Determine row background color based on R2Score
            r2_score = gpe_scores.get("R2Score", None)
            if r2_score is not None:
                r2_score = float(r2_score)
                if r2_score > 0.9:
                    pdf.set_fill_color(204, 255, 204)  # Pastel green
                elif 0.7 <= r2_score <= 0.9:
                    pdf.set_fill_color(255, 255, 204)  # Yellow
                else:
                    pdf.set_fill_color(255, 204, 204)  # Red
            else:
                pdf.set_fill_color(255, 255, 255)  # Default white

            # Create row for this output
            row = [ylabel]
            for column in columns:
                row.append(gpe_scores.get(column, "N/A"))

        # Add row to PDF
        for cell in row:
            pdf.cell(cell_width, 10, str(cell), border=1, align='C', fill=True)
        pdf.ln()

    # Output the PDF
    pdf.output(output_pdf_path)




In [ ]:
ylabels_path = f"{basefolder}/data/ylabels.txt"
output_pdf_path = f"{emulators_folder_base}/output_table.pdf"
create_pdf_with_table(ylabels_path, emulators_folder_base, output_pdf_path)